Bronze layer validation/checking cell

In [0]:
# Check columns and sample data

print("CUSTOMERS")
spark.table("ecommerce_data_analysis.bronze.bronze_customers").printSchema()
spark.table("ecommerce_data_analysis.bronze.bronze_customers").show(5)

print("PRODUCTS")
spark.table("ecommerce_data_analysis.bronze.bronze_products").printSchema()
spark.table("ecommerce_data_analysis.bronze.bronze_products").show(5)

print("SALES")
spark.table("ecommerce_data_analysis.bronze.bronze_sales").printSchema()
spark.table("ecommerce_data_analysis.bronze.bronze_sales").show(5)

CUSTOMERS
root
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Age_Group: string (nullable = true)
 |-- Date_of_Birth: date (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Pincode: integer (nullable = true)
 |-- Registration_Date: date (nullable = true)
 |-- Customer_Tier: string (nullable = true)
 |-- Total_Orders: integer (nullable = true)
 |-- Total_Spent: double (nullable = true)

+------------+-------------+------+---+---------+-------------+--------------------+--------------+--------+-----------+-------+-----------------+-------------+------------+-----------+
| Customer_ID|Customer_Name|Gender|Age|Age_Group|Date_of_Birth|               Email|         Phone|    City|      State|Pincode|Registration_Date|Customer_Tier|Total_Orders|Total_Spent|

Creating the silver schema in Databricks if not exist / if exist doesnt create it again and doesn't give an error

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS ecommerce_data_analysis.silver
""")

print("Silver schema created successfully")

Silver schema created successfully


Transforming the customer , product , sales tables by newly creating the silver delta table from source/bronze table

In [0]:
from pyspark.sql.functions import col, trim, lower, upper

customers = spark.table(
    "ecommerce_data_analysis.bronze.bronze_customers"
)

silver_customers = (
    customers
    .withColumn("Customer_ID", trim(col("Customer_ID")))
    .withColumn("Customer_Name", trim(col("Customer_Name")))
    .withColumn("Gender", upper(trim(col("Gender"))))
    .withColumn("Email", lower(trim(col("Email"))))
    .withColumn("Phone", trim(col("Phone")))
    .withColumn("City", trim(col("City")))
    .withColumn("State", trim(col("State")))
    .withColumn("Customer_Tier", trim(col("Customer_Tier")))
    .dropDuplicates(["Customer_ID"])
    .filter(col("Customer_ID").isNotNull())
)

silver_customers.write.mode("overwrite").format("delta").saveAsTable(
    "ecommerce_data_analysis.silver.silver_customers"
)

print("silver_customers created successfully")

silver_customers created successfully


In [0]:
from pyspark.sql.functions import col, trim

products = spark.table(
    "ecommerce_data_analysis.bronze.bronze_products"
)

silver_products = (
    products
    .withColumn("Product_ID", trim(col("Product_ID")))
    .withColumn("Product_Name", trim(col("Product_Name")))
    .withColumn("Category", trim(col("Category")))
    .withColumn("Brand", trim(col("Brand")))
    .dropDuplicates(["Product_ID"])
    .filter(col("Product_ID").isNotNull())
)

silver_products.write.mode("overwrite").format("delta").saveAsTable(
    "ecommerce_data_analysis.silver.silver_products"
)

print("silver_products created successfully")

silver_products created successfully


In [0]:
from pyspark.sql.functions import col, trim, upper

sales = spark.table(
    "ecommerce_data_analysis.bronze.bronze_sales"
)

silver_sales = (
    sales
    .withColumn("Order_ID", trim(col("Order_ID")))
    .withColumn("Customer_ID", trim(col("Customer_ID")))
    .withColumn("Product_ID", trim(col("Product_ID")))
    .withColumn("Order_Date", col("Order_Date").cast("date"))
    .withColumn("Delivery_Date", col("Delivery_Date").cast("date"))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("Unit_Price", col("Unit_Price").cast("double"))
    .withColumn("Order_Value", col("Order_Value").cast("double"))
    .withColumn("Shipping_Cost", col("Shipping_Cost").cast("double"))
    .withColumn("Coupon_Discount", col("Coupon_Discount").cast("double"))
    .withColumn("Total_Amount", col("Total_Amount").cast("double"))
    .withColumn("Coupon_Code", upper(trim(col("Coupon_Code"))))
    .withColumn("Payment_Mode", upper(trim(col("Payment_Mode"))))
    .withColumn("Order_Status", upper(trim(col("Order_Status"))))
    .withColumn("City", trim(col("City")))
    .withColumn("State", trim(col("State")))
    .withColumn("Rating", col("Rating").cast("double"))
    .withColumn("Review_Text", trim(col("Review_Text")))
    .dropDuplicates(["Order_ID"])
    .filter(col("Order_ID").isNotNull())
)

silver_sales.write.mode("overwrite").format("delta").saveAsTable(
    "ecommerce_data_analysis.silver.silver_sales"
)

print("silver_sales created successfully")

silver_sales created successfully


Creating the enriched sales table by joining the three cleaned silver tables , one table contains sales + customer + product information

In [0]:
from pyspark.sql import functions as F

# Load Silver tables
sales = spark.table("ecommerce_data_analysis.silver.silver_sales")
customers = spark.table("ecommerce_data_analysis.silver.silver_customers")
products = spark.table("ecommerce_data_analysis.silver.silver_products")

# Join Sales + Customers + Products
silver_sales_enriched = (
    sales.alias("s")
    .join(
        customers.alias("c"),
        F.col("s.Customer_ID") == F.col("c.Customer_ID"),
        "left"
    )
    .join(
        products.alias("p"),
        F.col("s.Product_ID") == F.col("p.Product_ID"),
        "left"
    )
    .select(
        # Sales
        F.col("s.Order_ID"),
        F.col("s.Customer_ID"),
        F.col("s.Product_ID"),
        F.col("s.Order_Date"),
        F.col("s.Order_Time"),
        F.col("s.Delivery_Date"),
        F.col("s.Quantity"),
        F.col("s.Unit_Price"),
        F.col("s.Order_Value"),
        F.col("s.Shipping_Cost"),
        F.col("s.Coupon_Code"),
        F.col("s.Coupon_Discount"),
        F.col("s.Total_Amount"),
        F.col("s.Payment_Mode"),
        F.col("s.Order_Status"),
        F.col("s.Rating"),
        F.col("s.Review_Text"),
        F.col("s.City"),
        F.col("s.State"),

        # Customer
        F.col("c.Customer_Name"),
        F.col("c.Gender"),
        F.col("c.Age"),
        F.col("c.Age_Group"),
        F.col("c.Customer_Tier"),

        # Product
        F.col("p.Product_Name"),
        F.col("p.Category"),
        F.col("p.Brand"),
        F.col("p.Original_Price"),
        F.col("p.Discount_Percent"),
        F.col("p.Selling_Price"),
        F.col("p.Avg_Rating"),
        F.col("p.Total_Reviews")
    )
)

# Save enriched Silver table
silver_sales_enriched.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(
        "ecommerce_data_analysis.silver.silver_sales_enriched"
    )

print("silver_sales_enriched created successfully")

silver_sales_enriched created successfully


Silver validation/checking

In [0]:
# Check the final Silver table

silver_final = spark.table(
    "ecommerce_data_analysis.silver.silver_sales_enriched"
)

print("Columns:", len(silver_final.columns))
print("Rows:", silver_final.count())

silver_final.printSchema()
silver_final.show(5, truncate=False)

Columns: 32
Rows: 250000
root
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Order_Time: time(6) (nullable = true)
 |-- Delivery_Date: date (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Unit_Price: double (nullable = true)
 |-- Order_Value: double (nullable = true)
 |-- Shipping_Cost: double (nullable = true)
 |-- Coupon_Code: string (nullable = true)
 |-- Coupon_Discount: double (nullable = true)
 |-- Total_Amount: double (nullable = true)
 |-- Payment_Mode: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Review_Text: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Age_Group: string (nullable = true)
 |-- Custome